# GAN + CGAN + VAE Pipeline (GPU-ready)
This notebook implements the full pipeline: preprocessing, VAE baseline trained on normal only, a class-conditional GAN (CGAN) to synthesize minority attack classes, creation of a balanced dataset, training a supervised classifier on balanced data, threshold tuning for the VAE, and GPU + mixed precision + checkpointing.

**Notes before running:**
- Place your cleaned data file `cleaned_cicids2017_ready.csv` in the same folder as this notebook.
- The dataset **must** have a `Label` column with `'BENIGN'` and attack labels.
- This notebook is designed to run on a machine with a CUDA-capable GPU and a compatible TensorFlow installation.
- The notebook saves intermediate artifacts: scalers, models, metrics, and plots to the working directory.


In [ ]:
# Check GPU and set mixed precision
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('Available GPUs:', gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print('Could not set memory growth:', e)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision policy:', mixed_precision.global_policy())


In [ ]:
# STEP0: Helpers (metrics, plots, save functions)
import numpy as np, pandas as pd, os, json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def save_metrics(path, metrics_dict):
    df = pd.DataFrame([metrics_dict])
    if os.path.exists(path):
        df.to_csv(path, mode='a', header=False, index=False)
    else:
        df.to_csv(path, index=False)

def plot_confusion(cm, classes=['Normal','Attack'], title='Confusion Matrix', fname=None):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title)
    if fname: plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()

def plot_roc(y_true, scores, fname=None):
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr,tpr)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0,1],[0,1], '--', color='gray')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend(loc='lower right')
    if fname: plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    return roc_auc


---
# SECTION 1 — Dataset loading & preprocessing
This cell loads `cleaned_cicids2017_ready.csv`, drops unwanted columns, handles nulls/duplicates, downsizes dtypes and saves cleaned output.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

# Load
infile = 'cleaned_cicids2017_ready.csv'
assert os.path.exists(infile), f"File {infile} not found. Place your cleaned CSV in the working directory."
print('Loading', infile)
df = pd.read_csv(infile, low_memory=False)
print('Original shape:', df.shape)

# Drop specified unnecessary columns if present
cols_to_drop = [
    'Flow ID', 'Fwd Header Length.1', 'Source IP', 'Src IP', 'Source Port', 'Src Port',
    'Destination IP', 'Dst IP', 'Destination Port', 'Dst Port', 'Timestamp', 'Unnamed: 0',
    'Inbound', 'SimillarHTTP'
]
cols_present = [c for c in cols_to_drop if c in df.columns]
if cols_present:
    df.drop(columns=cols_present, inplace=True)
    print('Dropped cols:', cols_present)
else:
    print('No specified columns to drop were present')

# Handle missing values: fill numeric with median, drop any remaining NA rows
num_missing = df.isnull().sum().sum()
print('Total missing before:', num_missing)
if num_missing>0:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            med = df[col].median()
            df[col].fillna(med, inplace=True)
    df.dropna(inplace=True)
print('Missing after cleaning:', df.isnull().sum().sum())

# Drop duplicates
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]
print('Dropped duplicates:', before-after)

# Downsize types: float64->float32, int64->int32, objects->category when appropriate
def optimize_dtypes(df):
    for col in df.columns:
        dt = df[col].dtype
        if dt == 'float64':
            df[col] = df[col].astype('float32')
        elif dt == 'int64':
            df[col] = df[col].astype('int32')
        elif dt == 'object':
            nunique = df[col].nunique()
            if nunique / len(df[col]) < 0.5:
                df[col] = df[col].astype('category')
    return df

df = optimize_dtypes(df)
print('Optimized dtypes counts:\n', df.dtypes.value_counts())

# Save cleaned dataset
cleaned_path = 'cleaned_ready_for_gan.csv'
df.to_csv(cleaned_path, index=False)
print('Saved cleaned dataset to', cleaned_path, 'shape', df.shape)


---
# SECTION 2 — Baseline VAE trained on NORMAL only
Train a VAE on `BENIGN` samples only and evaluate on mixed test set (normal + attacks). This is the proper unsupervised anomaly-detection setup.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('cleaned_ready_for_gan.csv')
assert 'Label' in df.columns, "Dataset must contain 'Label' column with 'BENIGN' and attack labels"
print('Loaded cleaned dataset for VAE baseline, shape:', df.shape)

num_df = df.select_dtypes(include=[np.number]).copy()
labels = df['Label'].copy()

scaler = MinMaxScaler()

normal_df = df[df['Label']=='BENIGN']
attack_df = df[df['Label']!='BENIGN']
print('Counts: normal=', len(normal_df), 'attacks=', len(attack_df))

X_norm = scaler.fit_transform(normal_df.select_dtypes(include=[np.number]).astype(np.float32))
X_att = scaler.transform(attack_df.select_dtypes(include=[np.number]).astype(np.float32))

X_train_norm, X_val_norm = train_test_split(X_norm, test_size=0.2, random_state=42)

orig_dim = X_train_norm.shape[1]
print('VAE input dim:', orig_dim)

class VAE(tf.keras.Model):
    def __init__(self, original_dim, latent_dim=16):
        super().__init__()
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(original_dim,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(latent_dim*2),
        ])
        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(original_dim, activation='sigmoid'),
        ])
    def encode(self,x):
        mlogv = self.encoder(x)
        mean, logvar = tf.split(mlogv, 2, axis=1)
        return mean, logvar
    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar*0.5) + mean
    def decode(self,z):
        return self.decoder(z)
    def call(self,x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        return self.decode(z), mean, logvar

vae = VAE(orig_dim, latent_dim=16)
optimizer = tf.keras.optimizers.Adam(1e-3)

def compute_loss(model, x):
    x = tf.cast(x, tf.float32)
    x_recon, mean, logvar = model(x)
    recon_loss = tf.reduce_mean(tf.square(x - x_recon), axis=1)
    kl_loss = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mean) - tf.exp(logvar), axis=1)
    return tf.reduce_mean(recon_loss + kl_loss)

batch_size = 128
train_ds = tf.data.Dataset.from_tensor_slices(X_train_norm.astype(np.float32)).shuffle(10000).batch(batch_size)
for epoch in range(20):
    epoch_loss = 0.0; steps = 0
    for batch in train_ds:
        with tf.GradientTape() as tape:
            loss = compute_loss(vae, batch)
        grads = tape.gradient(loss, vae.trainable_variables)
        optimizer.apply_gradients(zip(grads, vae.trainable_variables))
        epoch_loss += loss.numpy(); steps += 1
    print(f'[VAE baseline] Epoch {epoch+1} loss={epoch_loss/steps:.6f}')

def recon_errors(model, X):
    X = tf.cast(X, tf.float32)
    recon, _, _ = model(X)
    mse = np.mean(np.square(X - recon), axis=1)
    return mse

val_err = recon_errors(vae, X_val_norm.astype(np.float32))
thr = float(np.mean(val_err) + 3*np.std(val_err))
print('Baseline threshold (mean+3std on val):', thr)

n_sample = 5000
X_test_norm = X_train_norm[np.random.choice(X_train_norm.shape[0], min(n_sample, X_train_norm.shape[0]), replace=False)]
X_test_att = X_att[np.random.choice(X_att.shape[0], min(n_sample, X_att.shape[0]), replace=False)]
X_test = np.vstack([X_test_norm, X_test_att])
y_test = np.hstack([np.zeros(len(X_test_norm), dtype=int), np.ones(len(X_test_att), dtype=int)])

test_err = recon_errors(vae, X_test.astype(np.float32))
y_pred = (test_err > thr).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
print('[VAE baseline] Acc:', acc, 'Prec:', prec, 'Rec:', rec, 'F1:', f1)
plot_confusion(cm, classes=['Normal','Attack'], title='VAE Baseline Confusion Matrix', fname='step1_cm.png')
roc_auc = plot_roc(y_test, test_err, fname='step1_roc.png')
save_metrics('step1_metrics.csv', {'step':'vae_baseline','acc':acc,'prec':prec,'rec':rec,'f1':f1,'auc':roc_auc,'thr':thr})


---
# SECTION 3 — Train CGAN for minority attack synthesis (uses GPU)
Trains a class-conditional GAN on minority (rare) attack classes to generate synthetic attack samples. This step may take long depending on data size and GPU.


In [ ]:
import joblib
from collections import Counter
df = pd.read_csv('cleaned_ready_for_gan.csv')
num_df = df.select_dtypes(include=[np.number])
labels = df['Label'].values

unique_labels = sorted(df['Label'].unique())
label_map = {lab:i for i,lab in enumerate(unique_labels)}
inv_label_map = {v:k for k,v in label_map.items()}
y_all = np.array([label_map[l] for l in labels])

scaler = MinMaxScaler()
X_all = scaler.fit_transform(num_df.astype(np.float32))
joblib.dump(scaler, 'cgan_scaler.pkl')
joblib.dump(label_map, 'label_map.pkl')

cnt = Counter(y_all)
print('Class counts:', {inv_label_map[k]:v for k,v in cnt.items()})

threshold = int(np.percentile(list(cnt.values()), 20))
minority_classes = [cls for cls,c in cnt.items() if c <= threshold and inv_label_map[cls] != 'BENIGN']
print('Minority class ids:', minority_classes)

data_dim = X_all.shape[1]
latent_dim = 100
num_classes = len(label_map)

import tensorflow as tf
def make_generator(latent_dim, num_classes, data_dim):
    noise_in = tf.keras.layers.Input(shape=(latent_dim,))
    label_in = tf.keras.layers.Input(shape=(1,), dtype='int32')
    embed = tf.keras.layers.Embedding(num_classes, 50)(label_in)
    embed = tf.keras.layers.Flatten()(embed)
    x = tf.keras.layers.Concatenate()([noise_in, embed])
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    out = tf.keras.layers.Dense(data_dim, activation='sigmoid')(x)
    return tf.keras.Model([noise_in, label_in], out)

def make_discriminator(num_classes, data_dim):
    data_in = tf.keras.layers.Input(shape=(data_dim,))
    label_in = tf.keras.layers.Input(shape=(1,), dtype='int32')
    embed = tf.keras.layers.Embedding(num_classes, 50)(label_in)
    embed = tf.keras.layers.Flatten()(embed)
    x = tf.keras.layers.Concatenate()([data_in, embed])
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    return tf.keras.Model([data_in, label_in], out)

generator = make_generator(latent_dim, num_classes, data_dim)
discriminator = make_discriminator(num_classes, data_dim)
discriminator.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy')

noise_in = tf.keras.layers.Input(shape=(latent_dim,))
label_in = tf.keras.layers.Input(shape=(1,), dtype='int32')
gen_out = generator([noise_in, label_in])
discriminator.trainable = False
validity = discriminator([gen_out, label_in])
cgan = tf.keras.Model([noise_in, label_in], validity)
cgan.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy')

mask = np.isin(y_all, minority_classes)
if mask.sum() == 0:
    print('No minority classes found under threshold — adjust threshold manually or inspect class distribution')
X_min = X_all[mask]
y_min = y_all[mask]
print('Minority training shape:', X_min.shape)

batch_size = 256
epochs = 400
dataset = tf.data.Dataset.from_tensor_slices((X_min.astype(np.float32), y_min)).shuffle(10000).batch(batch_size)
for epoch in range(epochs):
    for real_batch, lbl_batch in dataset:
        bs = real_batch.shape[0]
        noise = np.random.normal(0,1,(bs, latent_dim))
        fake = generator.predict([noise, lbl_batch.reshape(-1,1)], verbose=0)
        X_comb = np.vstack((real_batch, fake))
        y_comb = np.vstack((np.ones((bs,1)), np.zeros((bs,1))))
        lbl_comb = np.vstack((lbl_batch.reshape(-1,1), lbl_batch.reshape(-1,1)))
        d_loss = discriminator.train_on_batch([X_comb, lbl_comb], y_comb)
        noise2 = np.random.normal(0,1,(bs, latent_dim))
        g_loss = cgan.train_on_batch([noise2, lbl_batch.reshape(-1,1)], np.ones((bs,1)))
    if epoch%50==0:
        print(f'[CGAN] Epoch {epoch}/{epochs}, d_loss={d_loss:.4f}, g_loss={g_loss:.4f}')

generator.save('cgan_generator.keras')
print('Saved CGAN generator to cgan_generator.keras and scaler, label_map saved earlier')


---
# SECTION 4 — Generate balanced dataset using CGAN
Use the trained CGAN generator to oversample minority attack classes to match the BENIGN class count.


In [ ]:
import joblib
from tensorflow.keras.models import load_model
from collections import Counter

scaler = joblib.load('cgan_scaler.pkl')
label_map = joblib.load('label_map.pkl')
inv_label_map = {v:k for k,v in label_map.items()}
generator = load_model('cgan_generator.keras', compile=False)

df = pd.read_csv('cleaned_ready_for_gan.csv')
num_df = df.select_dtypes(include=[np.number])
X_all = scaler.transform(num_df.astype(np.float32))
labels = df['Label'].map(label_map).values

cnt = Counter(labels)
normal_count = cnt[label_map['BENIGN']]
print('Normal count:', normal_count)

synthetic_samples = []
synthetic_labels = []

for cls_id, cls_count in cnt.items():
    cls_name = inv_label_map[cls_id]
    if cls_name == 'BENIGN':
        continue
    needed = max(0, normal_count - cls_count)
    if needed == 0:
        continue
    print(f'Generating {needed} samples for class', cls_name)
    batch = 2048
    gen_list = []
    while len(gen_list) < needed:
        b = min(batch, needed - len(gen_list))
        noise = np.random.normal(0,1,(b, generator.input_shape[0][1]))
        lbls = np.full((b,1), cls_id)
        gen = generator.predict([noise, lbls], verbose=0)
        gen_list.append(gen)
    gen_arr = np.vstack(gen_list)[:needed]
    synthetic_samples.append(gen_arr)
    synthetic_labels.extend([cls_id]*needed)

if synthetic_samples:
    synth_X = np.vstack(synthetic_samples)
    synth_y = np.array(synthetic_labels)
    synth_X_inv = scaler.inverse_transform(synth_X)
    synth_df = pd.DataFrame(synth_X_inv, columns=num_df.columns)
    synth_df['Label'] = [inv_label_map[i] for i in synth_y]
    balanced_df = pd.concat([df, synth_df], ignore_index=True)
else:
    balanced_df = df.copy()

balanced_df.to_csv('balanced_dataset.csv', index=False)
print('Saved balanced dataset to balanced_dataset.csv. New class counts:')
print(balanced_df['Label'].value_counts())


---
# SECTION 5 — Train supervised classifier on balanced dataset (MLP) and evaluate
Supervised training on balanced dataset typically yields high performance. We train a binary classifier (BENIGN vs Attack) as a demonstration.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df_bal = pd.read_csv('balanced_dataset.csv')
X = df_bal.select_dtypes(include=[np.number]).values
y = (df_bal['Label'] != 'BENIGN').astype(int).values

X_train, X_test_s, y_train, y_test_s = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler_std = StandardScaler().fit(X_train)
X_train_s = scaler_std.transform(X_train)
X_test_s = scaler_std.transform(X_test_s)

clf = MLPClassifier(hidden_layer_sizes=(256,128), early_stopping=True, max_iter=200)
clf.fit(X_train_s, y_train)
y_pred = clf.predict(X_test_s)
y_prob = clf.predict_proba(X_test_s)[:,1]

acc = accuracy_score(y_test_s, y_pred)
prec = precision_score(y_test_s, y_pred)
rec = recall_score(y_test_s, y_pred)
f1 = f1_score(y_test_s, y_pred)
auc = roc_auc_score(y_test_s, y_prob)
print('Supervised MLP on balanced data — Acc, Prec, Rec, F1, AUC')
print(acc, prec, rec, f1, auc)
save_metrics('step4b_metrics.csv', {'step':'mlp_balanced','acc':acc,'prec':prec,'rec':rec,'f1':f1,'auc':auc})
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_s, y_pred)
plot_confusion(cm, classes=['Normal','Attack'], title='MLP Balanced Confusion Matrix', fname='step4b_cm.png')


---
# SECTION 6 — VAE threshold tuning, checkpointing and mixed-precision training (GPU)
This section retrains (or continues training) the VAE with checkpointing and mixed precision. It also tunes threshold by validation and re-evaluates.


In [ ]:
import joblib
from sklearn.metrics import f1_score

scaler = joblib.load('cgan_scaler.pkl') if os.path.exists('cgan_scaler.pkl') else scaler
df_full = pd.read_csv('cleaned_ready_for_gan.csv')
normal_df = df_full[df_full['Label']=='BENIGN']
X_norm_all = scaler.transform(normal_df.select_dtypes(include=[np.number]).astype(np.float32))
X_train_norm, X_val_norm = train_test_split(X_norm_all, test_size=0.2, random_state=42)

orig_dim = X_train_norm.shape[1]
vae = VAE(orig_dim, latent_dim=16)
from tensorflow.keras import mixed_precision
opt = mixed_precision.LossScaleOptimizer(tf.keras.optimizers.Adam(1e-3))

ckpt = tf.train.Checkpoint(vae=vae, optimizer=opt)
manager = tf.train.CheckpointManager(ckpt, './vae_checkpoints', max_to_keep=5)
if manager.latest_checkpoint:
    ckpt.restore(manager.latest_checkpoint)
    print('Restored VAE from', manager.latest_checkpoint)
else:
    print('No checkpoint found, starting fresh')

batch_size = 128
train_ds = tf.data.Dataset.from_tensor_slices(X_train_norm.astype(np.float32)).shuffle(10000).batch(batch_size)
for epoch in range(50):
    epoch_loss = 0.0; steps = 0
    for batch in train_ds:
        with tf.GradientTape() as tape:
            loss = compute_loss(vae, batch)
        scaled_loss = opt.get_scaled_loss(loss)
        grads = tape.gradient(scaled_loss, vae.trainable_variables)
        grads = opt.get_unscaled_gradients(grads)
        opt.apply_gradients(zip(grads, vae.trainable_variables))
        epoch_loss += loss.numpy(); steps += 1
    print(f'[VAE CP] Epoch {epoch+1} loss={epoch_loss/steps:.6f}')
    if (epoch+1) % 5 == 0:
        manager.save()
        print('Saved checkpoint at epoch', epoch+1)

attack_df = df_full[df_full['Label']!='BENIGN']
X_val_att = scaler.transform(attack_df.select_dtypes(include=[np.number]).astype(np.float32))
X_val_mix = np.vstack([X_val_norm, X_val_att[:len(X_val_norm)]])
y_val_mix = np.hstack([np.zeros(len(X_val_norm)), np.ones(len(X_val_norm))])
val_errs = recon_errors(vae, X_val_mix.astype(np.float32))
th_list = np.linspace(val_errs.min(), val_errs.max(), 200)
best_f1 = 0; best_thr = None
for thr in th_list:
    y_pred_val = (val_errs > thr).astype(int)
    f1v = f1_score(y_val_mix, y_pred_val)
    if f1v > best_f1:
        best_f1 = f1v; best_thr = thr
print('Best threshold on val:', best_thr, 'best_f1:', best_f1)

if os.path.exists('balanced_dataset.csv'):
    df_bal = pd.read_csv('balanced_dataset.csv')
    X_test_bal = scaler.transform(df_bal.select_dtypes(include=[np.number]).astype(np.float32))
    y_test_bal = (df_bal['Label'] != 'BENIGN').astype(int).values
    test_errs_bal = recon_errors(vae, X_test_bal.astype(np.float32))
    y_pred_bal = (test_errs_bal > best_thr).astype(int)
    acc = accuracy_score(y_test_bal, y_pred_bal)
    prec = precision_score(y_test_bal, y_pred_bal)
    rec = recall_score(y_test_bal, y_pred_bal)
    f1 = f1_score(y_test_bal, y_pred_bal)
    print('[VAE final eval] Acc,Prec,Rec,F1:', acc, prec, rec, f1)
    save_metrics('vae_final_metrics.csv', {'step':'vae_final','acc':acc,'prec':prec,'rec':rec,'f1':f1,'thr':best_thr})
    cm = confusion_matrix(y_test_bal, y_pred_bal)
    plot_confusion(cm, classes=['Normal','Attack'], title='VAE Final Confusion Matrix', fname='vae_final_cm.png')
    plot_roc(y_test_bal, test_errs_bal, fname='vae_final_roc.png')
else:
    print('balanced_dataset.csv not found — run CGAN steps to generate balanced dataset first')


---
# END: Saved artifacts
This notebook saves artifacts (scalers, models, datasets, metrics, plots) to the working directory.
Run the cells sequentially and monitor the outputs.
